
# Splitting & reassigning Cellpose membrane labels using object centroid seeds

This notebook provides a workflow that:

- reads 3D membrane label volumes (integer labelled 16-bit TIFF stacks), i.e. results from Cellpose segmentation
- identifies unique membrane objects and finds their center slice,
- expands those objects across adjacent slices using an **IoU** threshold,
- **resolves conflicts** when multiple centroid objects claim the same membrane slice,
- reconstructs a final relabeled 3D mask,
- optionally, filters out objects that are only a single z-slice

---

**Dependencies**: `numpy`, `tifffile`, `scikit-image`, `joblib`, `tqdm`.  
**Input format**: Labeled membrane masks as 3D TIFF (axes order: Z, Y, X).  
**Output**: TIFF stacks for intermediate and final results.




## 1. Configuration
Set global parameters here. You can tune `IOU_THRESHOLD` to control the minimum IoU values for object expanding across Z.


In [1]:

# -*- coding: utf-8 -*-
from typing import Dict, List, Tuple, Optional

import os
import glob
import gc
import numpy as np
import tifffile as tiff
from skimage.measure import regionprops, label
from joblib import Parallel, delayed
import multiprocessing
from tqdm import tqdm

# Parameters
IOU_THRESHOLD: float = 0.60          # IoU required to adopt a membrane slice into an object's Z-extent
CONNECTIVITY: int = 2               # Connectivity for 2D labeling (per-slice) when relabeling
FINAL_DTYPE = np.int16              # dtype of output label stacks
USE_TQDM: bool = True               # Toggle progress bars
SAVE_INTERMEDIATE: bool = True      # Save optional intermediate TIFFs

# Helper for tqdm
_tqdm = tqdm if USE_TQDM else lambda x, **kwargs: x



## 2. Helper Functions
These functions implement each step of the pipeline in a readable, testable way.


In [ ]:

def filter_single_z_slice_objects(array: np.ndarray) -> np.ndarray:
    '''Remove objects whose labels appear in **exactly one** Z slice.

    Parameters
    ----------
    array : np.ndarray
        3D labeled volume (Z, Y, X).

    Returns
    -------
    np.ndarray
        3D labeled volume with single-Z-slice labels set to 0 (background).
    '''
    
    # Step 1: Count unique labels in each z-slice
    unique_labels_all_slices = np.array(0)
    for z in range(array.shape[0]):
        unique_labels_all_slices = np.append(unique_labels_all_slices,np.unique(array[z]))
    
    #Step 2: Identify labels that appear only once across all z-slices
    #print("finding labels that are only in 1 z slice")
    unique_values, value_counts = np.unique(unique_labels_all_slices, return_counts=True)
    non_duplicate_values = unique_values[value_counts == 1]

    print(f"Found {len(non_duplicate_values)} 1 slice only labels out of {len(unique_values)} total labels")

    # Step 3: Filter out objects with labels present in only one z-slice
    #print("copying original array")
    filtered_array = np.copy(array)
    # Create a mask where True indicates the positions where 'array' matches any value in 'non_duplicate_values'
    mask = np.isin(array, non_duplicate_values)
    
    # Set the elements in `array` where mask is True to 0
    filtered_array[mask] = 0
    return filtered_array


def calculate_iou(slice1: np.ndarray, slice2: np.ndarray) -> float:
    '''Compute Intersection-over-Union for two boolean masks.

    Both masks must be 2D arrays of the same shape (Y, X) representing a single Z slice.
    '''
    intersection = np.logical_and(slice1, slice2).sum()
    union = np.logical_or(slice1, slice2).sum()
    if union == 0:
        return 0
    return intersection / union


def build_centroid_slices(membrane: np.ndarray) -> Tuple[np.ndarray, Dict[int, Tuple[float, float, float]]]:
    '''Create an array containing the membrane object present at each object's **centroid Z**.

    Steps
    -----
    1) Label the full 3D membrane volume.
    2) For each labeled object, find its centroid (z, y, x).
    3) At that centroid location, read the original membrane label value and copy the corresponding **2D slice mask**
       (same object label at that Z) into the result array.

    Returns
    -------
    centroid_slices : np.ndarray
        3D volume where each object's membrane at its centroid Z is kept (other positions zero).
    centroid_dict : Dict[int, Tuple[float, float, float]]
        Mapping from object label -> (z, y, x) centroid.
    '''
    labeled_membrane = label(membrane)  # 3D labeling
    props = regionprops(labeled_membrane)

    centroid_slices = np.zeros_like(membrane, dtype=FINAL_DTYPE)
    centroid_dict: Dict[int, Tuple[float, float, float]] = {}

    for prop in _tqdm(props, desc="Centroids"):
        obj_label = prop.label
        zc, yc, xc = prop.centroid  # floats
        z = int(round(zc)); y = int(round(yc)); x = int(round(xc))
        centroid_dict[obj_label] = (zc, yc, xc)
        if 0 <= z < membrane.shape[0]:
            # Membrane label at the centroid location
            mem_val = membrane[z, y, x]
            if mem_val != 0:
                mask2d = (membrane[z] == mem_val)
                centroid_slices[z][mask2d] = obj_label
    return centroid_slices, centroid_dict


def expand_slices_for_object(
    label_id: int,
    z_centroid: int,
    y_centroid: int,
    x_centroid: int,
    centroid_slices_unique: np.ndarray,
    membrane: np.ndarray,
    iou_threshold: float,
) -> Tuple[int, List[Tuple[int, int]]]:
    '''Expand a centroid object's 2D mask across Z using IoU w.r.t. adjacent slices.

    Parameters
    ----------
    label_id : int
        Unique label for the centroid slice object (after per-slice relabeling).
    z_centroid, y_centroid, x_centroid : int
        Rounded centroid coordinates of the original 3D object.
    centroid_slices_unique : np.ndarray
        3D volume containing unique labels per Z for centroid-based masks.
    membrane : np.ndarray
        Original membrane-labeled 3D volume.
    iou_threshold : float
        Minimum IoU to adopt candidate slice.

    Returns
    -------
    (label_id, expanded_slices): Tuple[int, List[Tuple[int, int]]]
        `expanded_slices` is a list of (z, original_membrane_label_value) pairs owned by `label_id`.
    '''
    # Current object's 2D mask at the centroid Z
    current_mask = (centroid_slices_unique[z_centroid] == label_id)
    orig_label_at_centroid = membrane[z_centroid, y_centroid, x_centroid]

    expanded: List[Tuple[int, int]] = []
    if orig_label_at_centroid != 0:
        expanded.append((z_centroid, orig_label_at_centroid))

    # Try expanding up (-1) and down (+1)
    zmax = membrane.shape[0]
    for direction in (-1, 1):
        z = z_centroid + direction
        while 0 <= z < zmax:
            candidate_label_val = membrane[z, y_centroid, x_centroid]
            if candidate_label_val == 0:
                break  # no object at this coordinate
            candidate_mask = (membrane[z] == candidate_label_val)
            iou = calculate_iou(current_mask, candidate_mask)
            if iou > iou_threshold:
                expanded.append((z, candidate_label_val))
                z += direction
            else:
                break
    return label_id, expanded


def resolve_conflicts(
    expanded_objects: Dict[int, List[Tuple[int, int]]],
    centroid_slices_unique: np.ndarray,
    membrane: np.ndarray,
    centroid_dict: Dict[int, Tuple[float, float, float]],
) -> Dict[int, List[Tuple[int, int]]]:
    '''Resolve conflicts when multiple objects claim the **same** (z, membrane_label_value) slice.

    Strategy
    --------
    For a conflicting slice, prefer the owner whose centroid slice has the **higher IoU** with the candidate membrane slice.
    '''
    ownership: Dict[Tuple[int, int], int] = {}
    final: Dict[int, List[Tuple[int, int]]] = {}

    for label_id, slices in expanded_objects.items():
        final[label_id] = []
        for z, orig_val in slices:
            key = (z, orig_val)
            if key not in ownership:
                ownership[key] = label_id
                final[label_id].append((z, orig_val))
            else:
                current_owner = ownership[key]
                cz_curr = int(round(centroid_dict[current_owner][0]))
                cz_new = int(round(centroid_dict[label_id][0]))

                # IoU of owners' centroid masks with candidate slice in original membrane
                curr_mask = (centroid_slices_unique[cz_curr] == current_owner)
                new_mask  = (centroid_slices_unique[cz_new] == label_id)
                candidate = (membrane[z] == orig_val)

                iou_curr = calculate_iou(curr_mask, candidate)
                iou_new  = calculate_iou(new_mask,  candidate)

                if iou_new > iou_curr:
                    ownership[key] = label_id
                    final[label_id].append((z, orig_val))
                    # remove from previous owner's list if present
                    prev_list = final.get(current_owner, [])
                    final[current_owner] = [p for p in prev_list if p != (z, orig_val)]
                # else: keep current owner; new owner does not get this slice
    # Drop empty entries
    final = {lid: sl for lid, sl in final.items() if sl}
    return final


def reconstruct_final_array(
    final_expanded: Dict[int, List[Tuple[int, int]]],
    membrane: np.ndarray,
) -> np.ndarray:
    '''Build the final relabeled 3D array from ownership mapping.

    For each `(z, orig_label_val)` pair owned by `label_id`, assign all voxels with `orig_label_val` **at that Z**
    to `label_id` in the output.
    '''
    out = np.zeros_like(membrane, dtype=FINAL_DTYPE)
    for label_id, slices in final_expanded.items():
        for z, orig_val in slices:
            out[z][membrane[z] == orig_val] = label_id
    return out


def process_one_stack(
    membrane_path: str,
    save_dir: Optional[str] = None,
    iou_threshold: float = IOU_THRESHOLD,
    filter_single_slice: bool = True,
    save_intermediate: bool = SAVE_INTERMEDIATE,
) -> Dict[str, str]:
    '''Process a single 3D membrane label TIFF stack end-to-end.

    Parameters
    ----------
    membrane_path : str
        Path to input TIFF (Z, Y, X) with integer labels.
    save_dir : Optional[str]
        Directory to save outputs. Defaults to same directory as input.
    iou_threshold : float
        IoU threshold used during expansion.
    filter_single_slice : bool
        If True, remove objects present in exactly one Z slice at the end.
    save_intermediate : bool
        If True, save centroid-slice arrays and unique relabels.

    Returns
    -------
    Dict[str, str]
        Mapping of artifact names to saved file paths.
    '''
    if save_dir is None:
        save_dir = os.path.dirname(os.path.abspath(membrane_path))

    # Read input
    membrane = tiff.imread(membrane_path)
    assert membrane.ndim == 3, "Input must be a 3D stack (Z, Y, X)."

    # Step 1: Build centroid slices
    centroid_slices, centroid_dict = build_centroid_slices(membrane)

    if save_intermediate:
        tiff.imwrite(os.path.join(save_dir, "centroid_slices.tif"), centroid_slices.astype(FINAL_DTYPE))

    # Step 3: Expand (parallel per object)
    zyx: Dict[int, Tuple[int, int, int]] = {
        lid: (int(round(z)), int(round(y)), int(round(x)))
        for lid, (z, y, x) in centroid_dict.items()
    }

    num_cores = multiprocessing.cpu_count()
    jobs = (
        delayed(expand_slices_for_object)(
            lid, z, y, x, centroid_slices, membrane, iou_threshold
        )
        for lid, (z, y, x) in (_tqdm(zyx.items(), desc="Expand", total=len(zyx)) if USE_TQDM else zyx.items())
    )

    with Parallel(n_jobs=num_cores) as parallel:
        expanded_list = parallel(jobs)

    expanded_dict = dict(expanded_list)
    del expanded_list
    gc.collect()

    # Step 4: Resolve conflicts
    final_expanded = resolve_conflicts(expanded_dict, centroid_slices, membrane, centroid_dict)

    # Step 5: Reconstruct final array
    final_array = reconstruct_final_array(final_expanded, membrane)

    # Optional: remove single-slice objects
    if filter_single_slice:
        final_array = filter_single_z_slice_objects(final_array)

    # Save final
    out_paths: Dict[str, str] = {}
    final_name = os.path.splitext(os.path.basename(membrane_path))[0] + "_relabelled.tif"
    out_final = os.path.join(save_dir, final_name)
    tiff.imwrite(out_final, final_array.astype(FINAL_DTYPE))
    out_paths["final_relabelled"] = out_final

    if filter_single_slice:
        out_paths["filtered_single_slice"] = out_final  # same as final here because we overwrite `final_array`

    return out_paths


def batch_process(
    input_dir: str,
    pattern: str = "*_cp_masks.tif",
    output_base: Optional[str] = None,
    create_subfolders: bool = True,
    iou_threshold: float = IOU_THRESHOLD,
    filter_single_slice: bool = True,
    save_intermediate: bool = False,
) -> Dict[str, Dict[str, str]]:
    """
    Batch-process all membrane stacks matching `pattern` in `input_dir`.

    If `output_base` is provided, each file's outputs are written to a separate
    folder under `output_base` (or under `input_dir` when `output_base` is None).
    Subfolder name is "<basename>_results" by default (can be disabled with
    create_subfolders=False, which will write directly into the base output dir).
    Returns a mapping from input file -> the dictionary returned by process_one_stack.
    """

    paths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    if not paths:
        print("No files found.")
        return {}

    dest_base = output_base or input_dir
    os.makedirs(dest_base, exist_ok=True)

    results_map: Dict[str, Dict[str, str]] = {}
    for p in _tqdm(paths, desc="Files"):
        base_name = os.path.splitext(os.path.basename(p))[0]
        if create_subfolders:
            out_dir = os.path.join(dest_base, f"{base_name}_results")
        else:
            out_dir = dest_base
        os.makedirs(out_dir, exist_ok=True)

        try:
            out_paths = process_one_stack(
                membrane_path=p,
                save_dir=out_dir,
                iou_threshold=iou_threshold,
                filter_single_slice=filter_single_slice,
                save_intermediate=save_intermediate,
            )
            results_map[p] = out_paths
        except Exception as e:
            print(f"[ERROR] {p}: {e}")

    return results_map


In [ ]:


def batch_process(
    input_dir: str,
    pattern: str = "*_cp_masks.tif",
    output_base: Optional[str] = None,
    create_subfolders: bool = False,
    iou_threshold: float = IOU_THRESHOLD,
    filter_single_slice: bool = True,
    save_intermediate: bool = False,
) -> Dict[str, Dict[str, str]]:
    """
    Batch-process all membrane stacks matching `pattern` in `input_dir`.

    If `output_base` is provided, each file's outputs are written to a separate
    folder under `output_base` (or under `input_dir` when `output_base` is None).
    Subfolder name is "<basename>_results" by default (can be disabled with
    create_subfolders=False, which will write directly into the base output dir).
    Returns a mapping from input file -> the dictionary returned by process_one_stack.
    """

    paths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    if not paths:
        print("No files found.")
        return {}

    dest_base = output_base or input_dir
    os.makedirs(dest_base, exist_ok=True)

    results_map: Dict[str, Dict[str, str]] = {}
    for p in _tqdm(paths, desc="Files"):
        base_name = os.path.splitext(os.path.basename(p))[0]
        if create_subfolders:
            out_dir = os.path.join(dest_base, f"{base_name}_results")
        else:
            out_dir = dest_base
        os.makedirs(out_dir, exist_ok=True)

        try:
            out_paths = process_one_stack(
                membrane_path=p,
                save_dir=out_dir,
                iou_threshold=iou_threshold,
                filter_single_slice=filter_single_slice,
                save_intermediate=save_intermediate,
            )
            results_map[p] = out_paths
        except Exception as e:
            print(f"[ERROR] {p}: {e}")

    return results_map

# Example usage:
# batch_process("/path/to/input", output_base="/path/to/outputs", pattern="*_cp_masks.tif")


## 3. Single-file Example
Set `MEMBRANE_PATH` to your 3D membrane label stack (TIFF) and run. Outputs will be saved next to the input by default.


In [ ]:

# Example — edit this path before running
MEMBRANE_PATH = r"D:/Mari_Sixth_Dataset_Analysis/split_nuclei_membrane_raw/VollCellPoseSeg/CellPose2/Merged-359_cp_masks.tif"  # e.g. 'mem-mneongreen-tiltcorrected-360_cp_masks_stitch07.tif'

# Uncomment after setting a valid path
results = process_one_stack(
    membrane_path=MEMBRANE_PATH,
    save_dir=None,               # or provide an explicit output directory
    iou_threshold=IOU_THRESHOLD,
    filter_single_slice=True,
    save_intermediate=True,
)
results



## 4. Batch Processing Example
Process every file that matches a glob pattern in a folder (e.g., all `*_cp_masks.tif`).


In [ ]:

# Example — edit this directory before running
INPUT_DIR = r"/path/to/folder/with/masks"  # Folder containing many '*_cp_masks.tif' stacks
OUTPUT_DIR = r"/path/to/output/folder"    # Base output folder


# Uncomment after setting valid folders
# batch_process(
#     input_dir=INPUT_DIR,
#     pattern="*_cp_masks.tif",
#     output_base=OUTPUT_DIR,
#     create_subfolders = False,
#     iou_threshold=IOU_THRESHOLD,
#     filter_single_slice=True,
#     save_intermediate=False,
# )



---
### 5. Notes & Tips
- **Memory**: For very large volumes, ensure enough RAM; consider chunked I/O or downsampling if needed.
- **IoU threshold**: Higher values will reduce expansion across Z and keep objects more conservative.
-  This notebook was **cleaned and consolidated** using Microsoft's CoPilot AI, run and tested 2025-05-12 by Mari Tolonen.
